## 전역

In [ ]:
# ============================================================
# [SHAP 단독 실행 시 필요한 변수 재선언]
# ============================================================

import pandas as pd
import numpy as np
import os
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

# ── ★ 한글 폰트 설정 ──────────────────────────────────────
import platform

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"       # 윈도우
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"         # 맥
else:
    # Linux: 나눔고딕 설치 필요 (pip install koreanize-matplotlib)
    try:
        import koreanize_matplotlib
    except ImportError:
        pass

plt.rcParams["axes.unicode_minus"] = False              # 마이너스 기호 깨짐 방지

# ── 경로 설정 ─────────────────────────────────────────────
TRAIN_PATH = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH  = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'

TARGET_COL    = "부실라벨_ICR3년"
RANDOM_STATE  = 42
THRESHOLD     = 0.44
SHAP_SAVE_DIR = r"16번. SHAP\전역"      # ★ 경로 변경
os.makedirs(SHAP_SAVE_DIR, exist_ok=True)

# ── 데이터 로드 ───────────────────────────────────────────
train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

pos_weight = (y_train_full == 0).sum() / (y_train_full == 1).sum()

# ── 피처 파일 ─────────────────────────────────────────────
feature_files = {
    "top55_dedup45": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv",
}

# ── feature_map 재구성 ────────────────────────────────────
feature_map = {}
for feature_name, feature_path in feature_files.items():
    df_feat        = pd.read_csv(feature_path)
    col_key        = "feature" if "feature" in df_feat.columns else df_feat.columns[0]
    raw_features   = df_feat[col_key].tolist()
    valid_features = [f for f in raw_features if f in train_full.columns]
    feature_map[feature_name] = valid_features
    print(f"[{feature_name}]  피처: {len(valid_features)}개")

print(f"pos_weight: {pos_weight:.4f}")
print("=" * 70)


# ============================================================
# 모델 정의
# ============================================================

def make_final_model():
    return XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    )


# ============================================================
# 피처셋별 SHAP + Permutation 루프
# ============================================================

for feature_name, use_features in feature_map.items():

    feature_file_name = feature_files[feature_name].split("\\")[-1]
    print(f"\n{'='*70}")
    print(f"[SHAP/Permutation] Feature Set: {feature_name}  |  피처 수: {len(use_features)}개")
    print(f"{'='*70}")

    # ── 전체 train 재학습 ─────────────────────────────────────
    imputer_shap = SimpleImputer(strategy="median")
    X_train_shap = pd.DataFrame(
        imputer_shap.fit_transform(train_full[use_features]),
        columns=use_features
    )
    X_test_shap  = pd.DataFrame(
        imputer_shap.transform(test[use_features]),
        columns=use_features
    )

    model_shap = make_final_model()
    model_shap.fit(X_train_shap, y_train_full)

    # ── ★ 서브폴더: 16번. SHAP/전역/피처셋명 ─────────────────
    save_sub = os.path.join(SHAP_SAVE_DIR, feature_name)
    os.makedirs(save_sub, exist_ok=True)


    # ── [A-1] SHAP 계산 ───────────────────────────────────────
    print(f"  SHAP 계산 중...")
    explainer   = shap.TreeExplainer(model_shap)
    shap_values = explainer.shap_values(X_test_shap)

    shap_df = pd.DataFrame(shap_values, columns=use_features)
    shap_df.to_csv(
        os.path.join(save_sub, f"shap_values_{feature_name}.csv"),
        index=False, encoding="utf-8-sig"
    )

    # ── [A-2] Bar Plot ────────────────────────────────────────
    plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
    shap.summary_plot(shap_values, X_test_shap, plot_type="bar",
                      show=False, max_display=20)
    plt.title(f"SHAP Global Feature Importance (Bar)\n{feature_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(
        os.path.join(save_sub, f"shap_bar_{feature_name}.png"),
        dpi=150, bbox_inches="tight"
    )
    plt.close()

    # ── [A-3] Beeswarm Plot ───────────────────────────────────
    plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
    shap.summary_plot(shap_values, X_test_shap, plot_type="dot",
                      show=False, max_display=20)
    plt.title(f"SHAP Beeswarm Plot\n{feature_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(
        os.path.join(save_sub, f"shap_beeswarm_{feature_name}.png"),
        dpi=150, bbox_inches="tight"
    )
    plt.close()

    # ── [A-4] SHAP 중요도 CSV ─────────────────────────────────
    shap_importance = pd.DataFrame({
        "Feature"       : use_features,
        "mean_abs_SHAP" : np.abs(shap_values).mean(axis=0)
    }).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)
    shap_importance["Rank"] = shap_importance.index + 1
    shap_importance.to_csv(
        os.path.join(save_sub, f"shap_importance_{feature_name}.csv"),
        index=False, encoding="utf-8-sig"
    )
    print(f"  [SHAP Top 10]\n{shap_importance.head(10).to_string(index=False)}")


    # ── [B-1] Permutation Importance ─────────────────────────
    print(f"\n  Permutation Importance 계산 중...")
    perm_result = permutation_importance(
        model_shap, X_test_shap, y_test,
        n_repeats=30, random_state=RANDOM_STATE,
        scoring="average_precision", n_jobs=-1
    )

    perm_df = pd.DataFrame({
        "Feature"   : use_features,
        "Perm_Mean" : perm_result.importances_mean,
        "Perm_Std"  : perm_result.importances_std,
    }).sort_values("Perm_Mean", ascending=False).reset_index(drop=True)
    perm_df["Rank"] = perm_df.index + 1
    perm_df.to_csv(
        os.path.join(save_sub, f"permutation_importance_{feature_name}.csv"),
        index=False, encoding="utf-8-sig"
    )
    print(f"  [Permutation Top 10]\n{perm_df.head(10).to_string(index=False)}")

    # ── [B-2] Permutation Bar Plot ────────────────────────────
    top_n    = min(20, len(use_features))
    perm_top = perm_df.head(top_n).sort_values("Perm_Mean", ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.4)))
    ax.barh(
        perm_top["Feature"], perm_top["Perm_Mean"],
        xerr=perm_top["Perm_Std"],
        color="#2F6EBA", alpha=0.8,
        error_kw=dict(ecolor="#555555", capsize=3)
    )
    ax.axvline(0, color="red", linestyle="--", linewidth=1.0)
    ax.set_xlabel("Mean decrease in PR_AUC", fontsize=10)
    ax.set_title(f"Permutation Importance (Top {top_n})\n{feature_name}", fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(
        os.path.join(save_sub, f"permutation_bar_{feature_name}.png"),
        dpi=150, bbox_inches="tight"
    )
    plt.close()

    # ── [C] SHAP vs Permutation 순위 비교 CSV ─────────────────
    compare_df = shap_importance[["Rank", "Feature", "mean_abs_SHAP"]].rename(
        columns={"Rank": "SHAP_Rank"}
    ).merge(
        perm_df[["Feature", "Perm_Mean", "Rank"]].rename(
            columns={"Rank": "Perm_Rank"}
        ),
        on="Feature", how="inner"
    )
    compare_df["Rank_Diff"] = (
        compare_df["SHAP_Rank"] - compare_df["Perm_Rank"]
    ).abs()
    compare_df = compare_df.sort_values("SHAP_Rank").reset_index(drop=True)
    compare_df.to_csv(
        os.path.join(save_sub, f"shap_vs_permutation_{feature_name}.csv"),
        index=False, encoding="utf-8-sig"
    )
    print(f"\n  → {save_sub} 저장 완료")


print("\n" + "=" * 70)
print("SHAP / Permutation Importance 분석 전체 완료")
print(f"  저장 위치: {SHAP_SAVE_DIR}")
print("=" * 70)

In [ ]:
# ============================================================
# SHAP vs Permutation 통합 분석 CSV 생성 (피처 범주 추가)
# ============================================================

import pandas as pd
import numpy as np
import os

SHAP_SAVE_DIR = r"16번. SHAP\전역"

# ============================================================
# ★ 피처 범주 사전 정의
# ============================================================

FEATURE_CATEGORY = {
    # 안정성 (Solvency)
    "자본잠식률"                        : "안정성 (Solvency)",
    "비유동장기적합률_ratio"             : "안정성 (Solvency)",
    "차입금의존도_diff_industry"         : "안정성 (Solvency)",
    "부채비율"                          : "안정성 (Solvency)",
    "자기자본비율_diff_industry"         : "안정성 (Solvency)",
    "부채비율변화"                       : "안정성 (Solvency)",
    "장기부채의존도"                     : "안정성 (Solvency)",
    "유동비율변화_diff"                  : "안정성 (Solvency)",
    "유동비율_ratio"                     : "안정성 (Solvency)",
    "장기부채비율"                       : "안정성 (Solvency)",
    "유보율_diff"                        : "안정성 (Solvency)",
    "순운전자본비율_ratio"               : "안정성 (Solvency)",
    "순운전자본대총자본_ratio_industry"  : "안정성 (Solvency)",

    # 수익성 (Profitability)
    "총자본영업이익률_diff"              : "수익성 (Profitability)",
    "금융비용부담률"                     : "수익성 (Profitability)",
    "ROA변화"                           : "수익성 (Profitability)",
    "매출액순이익률_diff_industry"       : "수익성 (Profitability)",
    "ROIC_diff"                         : "수익성 (Profitability)",
    "ROA_ratio"                         : "수익성 (Profitability)",
    "순이익률_ratio"                     : "수익성 (Profitability)",
    "현금ROA"                           : "수익성 (Profitability)",
    "매출총이익률_diff"                  : "수익성 (Profitability)",
    "매출원가율"                         : "수익성 (Profitability)",
    "ROE_diff"                          : "수익성 (Profitability)",
    "현금ROE_ratio"                     : "수익성 (Profitability)",

    # 성장성 (Growth)
    "매출액증가율"                       : "성장성 (Growth)",
    "순이익증가율_diff"                  : "성장성 (Growth)",
    "유형자산증가율_ratio_industry"      : "성장성 (Growth)",
    "총자산증가율_diff_industry"         : "성장성 (Growth)",
    "자기자본증가율"                     : "성장성 (Growth)",

    # 활동성 (Activity)
    "비유동자산회전율_ratio"             : "활동성 (Activity)",
    "유형자산회전율_diff"                : "활동성 (Activity)",
    "매출채권회전율_ratio_industry"      : "활동성 (Activity)",
    "순운전자본회전율_diff"              : "활동성 (Activity)",
    "유동자산회전율"                     : "활동성 (Activity)",
    "총자산회전율_ratio"                 : "활동성 (Activity)",
    "재고자산보유기간_ratio"             : "활동성 (Activity)",
    "매입채무지급기간_diff"              : "활동성 (Activity)",

    # 현금흐름 (Cash Flow)
    "영업CF_유동부채_diff"               : "현금흐름 (Cash Flow)",
    "영업CF_총부채_diff"                 : "현금흐름 (Cash Flow)",
    "FCF_총자산_ratio"                   : "현금흐름 (Cash Flow)",
    "영업현금흐름비율"                   : "현금흐름 (Cash Flow)",
    "감가상각비율"                       : "현금흐름 (Cash Flow)",

    # 기타
    "업력"                              : "기타 (Other)",
    "유형자산비율"                       : "기타 (Other)",
}

CATEGORY_ORDER = {
    "안정성 (Solvency)"        : 1,
    "수익성 (Profitability)"   : 2,
    "성장성 (Growth)"          : 3,
    "활동성 (Activity)"        : 4,
    "현금흐름 (Cash Flow)"     : 5,
    "기타 (Other)"             : 6,
    "미분류"                   : 7,
}


# ============================================================
# 피처셋별 루프
# ============================================================

for feature_name in feature_map.keys():

    save_sub = os.path.join(SHAP_SAVE_DIR, feature_name)

    # ── 파일 로드 ─────────────────────────────────────────
    shap_imp = pd.read_csv(
        os.path.join(save_sub, f"shap_importance_{feature_name}.csv")
    )
    perm_imp = pd.read_csv(
        os.path.join(save_sub, f"permutation_importance_{feature_name}.csv")
    )

    n_features = len(shap_imp)

    # ── 병합 ──────────────────────────────────────────────
    df = shap_imp[["Rank", "Feature", "mean_abs_SHAP"]].rename(
        columns={"Rank": "SHAP_Rank"}
    ).merge(
        perm_imp[["Feature", "Perm_Mean", "Perm_Std", "Rank"]].rename(
            columns={"Rank": "Perm_Rank"}
        ),
        on="Feature", how="inner"
    )

    # ── 정규화 순위 ───────────────────────────────────────
    df["SHAP_Rank_Norm"] = (df["SHAP_Rank"] - 1) / (n_features - 1)
    df["Perm_Rank_Norm"] = (df["Perm_Rank"] - 1) / (n_features - 1)

    # ── 순위 차이 ─────────────────────────────────────────
    df["Rank_Diff"] = (df["SHAP_Rank"] - df["Perm_Rank"]).abs()

    # ── 통합 중요도 점수 ──────────────────────────────────
    df["Combined_Score"] = (df["SHAP_Rank_Norm"] + df["Perm_Rank_Norm"]) / 2
    df["Combined_Rank"]  = df["Combined_Score"].rank(method="min").astype(int)

    # ── 피처 유형 분류 ────────────────────────────────────
    top_n    = max(3, int(n_features * 0.3))
    shap_top = set(df.nsmallest(top_n, "SHAP_Rank")["Feature"])
    perm_top = set(df.nsmallest(top_n, "Perm_Rank")["Feature"])

    def classify(row):
        in_shap = row["Feature"] in shap_top
        in_perm = row["Feature"] in perm_top
        if in_shap and in_perm:
            return "★ 핵심피처 (SHAP+Perm 모두 높음)"
        elif in_shap and not in_perm:
            return "△ 대체가능 (SHAP 높음, Perm 낮음)"
        elif not in_shap and in_perm:
            return "▲ 상호작용 (Perm 높음, SHAP 낮음)"
        else:
            return "- 일반피처"

    df["Feature_Type"] = df.apply(classify, axis=1)

    # ── 불일치 레벨 ───────────────────────────────────────
    def rank_diff_level(diff):
        if diff <= 3:
            return "일치"
        elif diff <= 8:
            return "소폭 불일치"
        else:
            return "대폭 불일치"

    df["Consistency"] = df["Rank_Diff"].apply(rank_diff_level)

    # ── 범주 컬럼 추가 ────────────────────────────────────
    df["Category"]       = df["Feature"].map(FEATURE_CATEGORY).fillna("미분류")
    df["Category_Order"] = df["Category"].map(CATEGORY_ORDER)

    # ── 컬럼 순서 정리 & 정렬 ─────────────────────────────
    df = df[[
        "Combined_Rank",
        "Feature",
        "Category",
        "Feature_Type",
        "SHAP_Rank", "mean_abs_SHAP",
        "Perm_Rank", "Perm_Mean", "Perm_Std",
        "Rank_Diff", "Consistency",
        "Combined_Score",
        "Category_Order",
    ]].sort_values("Combined_Rank").reset_index(drop=True)

    # ── 저장 ──────────────────────────────────────────────
    out_path = os.path.join(save_sub, f"feature_analysis_{feature_name}.csv")
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    # ── 요약 출력 ─────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"[{feature_name}]  피처 분석 완료  →  {out_path}")
    print(f"{'='*70}")

    # 피처 유형 분포
    print(f"\n  [피처 유형 분포]")
    for t, c in df["Feature_Type"].value_counts().items():
        print(f"    {t} : {c}개")

    # ── ★ 범주별 분포 (수정된 부분) ──────────────────────
    print(f"\n  [범주별 분포]")
    cat_order_df = df[["Category", "Category_Order"]].drop_duplicates()

    cat_summary = (
        df.groupby("Category")
        .agg(
            피처수             = ("Feature",      "count"),
            평균_Combined_Rank = ("Combined_Rank", "mean"),
            핵심피처수         = ("Feature_Type",
                                  lambda x: (x == "★ 핵심피처 (SHAP+Perm 모두 높음)").sum())
        )
        .reset_index()
        .merge(cat_order_df, on="Category", how="left")   # ★ merge 먼저
        .sort_values("Category_Order")                    # ★ 그 다음 정렬
        .reset_index(drop=True)
    )
    print(cat_summary[[
        "Category", "피처수", "평균_Combined_Rank", "핵심피처수"
    ]].to_string(index=False))

    # 핵심피처 목록
    print(f"\n  [★ 핵심피처 목록 (SHAP+Perm 모두 상위 {top_n}위 이내)]")
    core = df[df["Feature_Type"] == "★ 핵심피처 (SHAP+Perm 모두 높음)"]
    if len(core) > 0:
        print(core[[
            "Combined_Rank", "Feature", "Category",
            "SHAP_Rank", "Perm_Rank", "Rank_Diff"
        ]].to_string(index=False))
    else:
        print("    없음")

    # 미분류 피처 경고
    unclassified = df[df["Category"] == "미분류"]
    if len(unclassified) > 0:
        print(f"\n  [⚠️ 미분류 피처 ({len(unclassified)}개) — FEATURE_CATEGORY 사전에 추가 필요]")
        print(unclassified[["Feature", "SHAP_Rank", "Perm_Rank"]].to_string(index=False))

    # 대폭 불일치 피처
    print(f"\n  [대폭 불일치 피처 (Rank_Diff > 8)]")
    mismatch = df[df["Consistency"] == "대폭 불일치"].sort_values(
        "Rank_Diff", ascending=False
    )
    if len(mismatch) > 0:
        print(mismatch[[
            "Feature", "Category", "Feature_Type",
            "SHAP_Rank", "Perm_Rank", "Rank_Diff"
        ]].to_string(index=False))
    else:
        print("    없음")

print("\n" + "=" * 70)
print("전체 피처 분석 CSV 저장 완료")
print("=" * 70)

## 지역

In [ ]:
# ============================================================
# [단독 실행용 변수 재선언]
# ============================================================

import shap
import matplotlib
import matplotlib as mpl
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import platform
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

# ── 한글 폰트 ─────────────────────────────────────────────
if platform.system() == "Windows":
    plt.rcParams["font.family"]        = "Malgun Gothic"
    mpl.rcParams["font.family"]        = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"]        = "AppleGothic"
    mpl.rcParams["font.family"]        = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
mpl.rcParams["axes.unicode_minus"] = False

# ── 경로 ──────────────────────────────────────────────────
TRAIN_PATH    = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH     = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
TARGET_COL    = "부실라벨_ICR3년"
YEAR_COL      = "회계년도"
COMPANY_COL   = "회사명"
RANDOM_STATE  = 42
THRESHOLD     = 0.44
SHAP_SAVE_DIR = r"16번. SHAP\지역"
os.makedirs(SHAP_SAVE_DIR, exist_ok=True)

# ── 데이터 로드 ───────────────────────────────────────────
train_full   = pd.read_parquet(TRAIN_PATH)
test         = pd.read_parquet(TEST_PATH)
y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]
pos_weight   = (y_train_full == 0).sum() / (y_train_full == 1).sum()
all_data     = pd.concat([train_full, test], ignore_index=True)

# ── 피처 파일 ─────────────────────────────────────────────
feature_files = {
    "top55_dedup45": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv",
}

# ── feature_map 재구성 ────────────────────────────────────
feature_map = {}
for feature_name, feature_path in feature_files.items():
    df_feat        = pd.read_csv(feature_path)
    col_key        = "feature" if "feature" in df_feat.columns else df_feat.columns[0]
    raw_features   = df_feat[col_key].tolist()
    valid_features = [f for f in raw_features if f in train_full.columns]
    feature_map[feature_name] = valid_features
    print(f"[{feature_name}]  피처: {len(valid_features)}개")

print(f"pos_weight : {pos_weight:.4f}")
print(f"전체 데이터: {len(all_data)}행  |  기업 수: {all_data[COMPANY_COL].nunique()}개")
print("=" * 70)


# ============================================================
# 조회할 기업 × 회계년도 목록 ← ★ 여기만 수정하면 됨
# ============================================================

QUERY_LIST = [
    {"회사명": "현대코퍼레이션(주)", "회계년도": 2021},
    {"회사명": "농협경제지주주식회사", "회계년도": 2021},
    # {"회사명": "기업명", "회계년도": 연도},
]


# ============================================================
# 헬퍼 함수
# ============================================================

def make_model():
    return XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    )


def get_sample(company, year):
    mask   = (all_data[COMPANY_COL] == company) & (all_data[YEAR_COL] == year)
    result = all_data[mask]
    if len(result) == 0:
        print(f"  ⚠️  [{company} / {year}] 데이터 없음")
        return None
    if len(result) > 1:
        print(f"  ⚠️  [{company} / {year}] 중복 {len(result)}행 → 첫 번째 행 사용")
    return result.iloc[[0]]


def fix_minus(fig):
    """유니코드 마이너스(−) → 일반 하이픈(-) 치환"""
    for ax in fig.get_axes():
        for t in ax.get_xticklabels():
            t.set_text(t.get_text().replace("\u2212", "-"))
        ax.set_xticklabels([t.get_text() for t in ax.get_xticklabels()])
        for t in ax.get_yticklabels():
            t.set_text(t.get_text().replace("\u2212", "-"))
        ax.set_yticklabels([t.get_text() for t in ax.get_yticklabels()])
        for txt in ax.texts:
            txt.set_text(txt.get_text().replace("\u2212", "-"))


# ============================================================
# 피처셋별 루프
# ============================================================

for feature_name, use_features in feature_map.items():

    print(f"\n{'='*70}")
    print(f"[로컬 SHAP] Feature Set: {feature_name}  |  피처 수: {len(use_features)}개")
    print(f"{'='*70}")

    save_sub = os.path.join(SHAP_SAVE_DIR, feature_name)
    os.makedirs(save_sub, exist_ok=True)

    # ── 전체 train으로 모델 학습 ──────────────────────────
    imputer     = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(train_full[use_features]),
        columns=use_features
    )
    model = make_model()
    model.fit(X_train_imp, y_train_full)

    # ── SHAP explainer 준비 ───────────────────────────────
    explainer = shap.TreeExplainer(model)

    # ── 전체 데이터 imputation ────────────────────────────
    all_feat_imp = pd.DataFrame(
        imputer.transform(all_data[use_features]),
        columns=use_features,
        index=all_data.index
    )

    # ── 쿼리별 분석 ───────────────────────────────────────
    for query in QUERY_LIST:

        company = query["회사명"]
        year    = query["회계년도"]

        print(f"\n  [{company} / {year}년]")

        sample_raw = get_sample(company, year)
        if sample_raw is None:
            continue

        sample_idx = sample_raw.index[0]
        X_sample   = all_feat_imp.loc[[sample_idx], use_features]

        y_true = sample_raw[TARGET_COL].values[0]
        y_prob = model.predict_proba(X_sample)[0, 1]
        y_pred = int(y_prob >= THRESHOLD)

        print(f"    실제 라벨 : {'부실(1)' if y_true == 1 else '정상(0)'}  |  "
              f"예측 확률 : {y_prob:.4f}  |  "
              f"예측 라벨 : {'부실(1)' if y_pred == 1 else '정상(0)'}")

        sv           = explainer(X_sample)
        shap_vals_1d = sv.values[0]
        base_value   = sv.base_values[0]

        safe_company = company.replace("/", "_").replace(" ", "_")
        save_company = os.path.join(save_sub, f"{safe_company}_{year}")
        os.makedirs(save_company, exist_ok=True)

        # ── [1] Waterfall Plot ────────────────────────────
        # shap 호출 직전 폰트 설정 강제 재적용
        mpl.rcParams["font.family"]        = "Malgun Gothic" \
                                             if platform.system() == "Windows" \
                                             else "AppleGothic"
        mpl.rcParams["axes.unicode_minus"] = False

        plt.figure(figsize=(10, max(6, len(use_features) * 0.28)))
        shap.plots.waterfall(sv[0], max_display=20, show=False)

        # 마이너스 기호 강제 치환
        fix_minus(plt.gcf())

        plt.title(
            f"SHAP Waterfall — {company} ({year}년)\n"
            f"실제: {'부실' if y_true==1 else '정상'}  |  "
            f"예측확률: {y_prob:.4f}  |  임계값: {THRESHOLD}",
            fontsize=11
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(save_company, f"waterfall_{safe_company}_{year}.png"),
            dpi=150, bbox_inches="tight"
        )
        plt.close()
        print(f"    → waterfall_{safe_company}_{year}.png 저장 완료")

        # ── [2] SHAP 기여도 CSV ───────────────────────────
        shap_row = pd.DataFrame({
            "Feature"    : use_features,
            "Value"      : X_sample.iloc[0].values,
            "SHAP_Value" : shap_vals_1d,
        })
        shap_row["Direction"] = shap_row["SHAP_Value"].apply(
            lambda x: "부실기여(+)" if x > 0 else "정상기여(-)"
        )
        shap_row["abs_SHAP"] = shap_row["SHAP_Value"].abs()
        shap_row = shap_row.sort_values(
            "abs_SHAP", ascending=False
        ).reset_index(drop=True)
        shap_row["Rank"]       = shap_row.index + 1
        shap_row.insert(0, "회계년도", year)
        shap_row.insert(0, "회사명",   company)
        shap_row["base_value"] = round(base_value, 6)
        shap_row["pred_prob"]  = round(y_prob, 6)
        shap_row["pred_label"] = y_pred
        shap_row["true_label"] = int(y_true)
        shap_row["threshold"]  = THRESHOLD

        shap_row = shap_row[[
            "회사명", "회계년도",
            "Rank", "Feature", "Value", "SHAP_Value",
            "Direction", "abs_SHAP",
            "base_value", "pred_prob", "pred_label",
            "true_label", "threshold"
        ]]
        shap_row.to_csv(
            os.path.join(save_company, f"shap_local_{safe_company}_{year}.csv"),
            index=False, encoding="utf-8-sig"
        )
        print(f"    → shap_local_{safe_company}_{year}.csv 저장 완료")

        # 상위 5개 출력
        print(f"\n    [상위 5개 기여 피처]")
        print(shap_row[[
            "Rank", "Feature", "Value", "SHAP_Value", "Direction"
        ]].head(5).to_string(index=False))


print("\n" + "=" * 70)
print("로컬 SHAP 분석 전체 완료")
print(f"  저장 위치: {SHAP_SAVE_DIR}")
print("=" * 70)